# 19a · **학습** — `transfer` 4 seed 한 번에 (4-GPU 노드)

seed 4개를 **동시에** 학습한다(GPU 4장). eval 은 `19b_eval_all` 에서.

- 그룹/ task 는 아래 `TAGS` · `TASK` 로 고른다 (기본: **`ours` · `transfer`**).
- 개별 노트북(`01`·`03`·`05`·`07`)과 **같은 함수**를 부르므로 결과는 동일하다.
- 끊겨도 안전: `--resume` 자동, 끝난 잡은 skip.

⚠️ GPU 4장짜리 노드용. 2장짜리 노드는 `12a`/`12b` 처럼 쪼갠 노트북을 쓴다.
⚠️ 학습 전에 **parity 테스트** — `00_smoke` 또는 `python tests/test_acm_sscp_literal.py`.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

# ── 무엇을 / 어디서 ──────────────────────────────────────────────────────────
TASK = cf.SHORT_SIM         # ★ 'transfer' (AlohaTransferCube-v0).  insertion 은 cf.MAIN_SIM
TAGS = cf.GROUP_OURS        # ['ours'] — 우리 모델
# TAGS = cf.GROUP_ACM       #   ['acm'] — 대조군(우리 주장의 분모)
# TAGS = cf.GROUP_OURS + cf.GROUP_ACM      #   둘 다 (8잡)
# TAGS = cf.GROUP_BASELINE  #   act·diffusion·smolvla·acm2 (16잡)
# TAGS = cf.GROUP_ABLATION  #   acm_carry·acm_bimamba·acm_s7 (12잡)

SEEDS = cf.MAIN_SEEDS       # [0,1,2,3] — 4 seed 동시
GPUS  = cf.v23.available_gpus()

print('task :', TASK, cf.v23.TASKS[TASK])
print('모델 :', TAGS, '| seeds:', SEEDS, '| GPU:', GPUS)
print('학습 : %s step, lr 고정, 학습중 eval %s (0=OFF)' % (f'{cf.STEPS:,}', cf.v23.EVAL_FREQ))
print('잡   :', len(TAGS) * len(SEEDS))

## 커맨드 확인 (dry-run)

In [ ]:
for t in TAGS:
    c = cf.make_train_cmd(t, seed=SEEDS[0], task=TASK, gpu_id=GPUS[0])
    print(f'{t:<10}', ' '.join(p for p in c.split()
                               if p.startswith(('CUDA_VISIBLE_DEVICES', '--dataset.repo_id',
                                                '--env.task', '--steps', '--policy.optimizer_lr'))))

## 학습 — 4 seed 동시 (resume 자동)
첫 실행은 데이터셋을 한 번 먼저 받는다(prefetch). 안 그러면 잡들이 같은 HF 캐시에
동시 다운로드를 걸어 대부분 죽는다.

In [ ]:
jobs = cf.run_training(TAGS, SEEDS, task=TASK, gpus=GPUS)

## 상태 — 150k 도달 확인

In [ ]:
cf.print_training_status(jobs)
print()
ok = cf.print_ckpt_status(TAGS, SEEDS, TASK)
print('\n=>', '다음: 19b_eval_all' if ok else '⚠️ 아직 150k 안 된 것이 있다')